# FlashMonarchAttention GPU benchmark

**Before running anything:** go to `Runtime` -> `Change runtime type` in the menu above, set **Hardware accelerator** to **GPU**, then click Save. Colab's free tier gives you a T4 GPU (a paid tier / Colab Pro can give you something faster, but T4 is enough to answer the question here).

**What this tests:** whether fusing small attention computations (FlashAttention-style, avoiding materializing intermediate score tensors) actually helps on GPU. On CPU we tested this and found no real win once dispatch overhead is controlled for -- the initial "speedup" we saw turned out to be a measurement artifact once isolated properly. GPU has a different memory hierarchy (HBM <-> SRAM) than CPU, so the answer might genuinely differ -- this notebook is how we check.

Run the cells below top to bottom (Shift+Enter on each, or `Runtime` -> `Run all`). Takes a couple of minutes. When it's done, copy the full output back.

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print()
    print("!! No GPU detected. Go to Runtime -> Change runtime type -> GPU, then re-run this cell. !!")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (PyTorch build): {torch.version.cuda}")

In [ ]:
import sys
import time
from math import sqrt
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "No GPU -- set Runtime > Change runtime type > GPU first, then re-run from the top."
device = torch.device("cuda")


def naive_attention(q, k, v, sm_scale, causal: bool):
    scores = sm_scale * (q @ k.transpose(-1, -2))
    if causal:
        n = q.shape[-2]
        mask = torch.tril(torch.ones(n, n, dtype=torch.bool, device=q.device))
        scores = scores.masked_fill(~mask, -float("inf"))
    probs = torch.softmax(scores, dim=-1)
    return probs @ v


def fused_attention(q, k, v, sm_scale, causal: bool):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, scale=sm_scale)


def bench(fn, *args, device, reps=50, warmup=8):
    with torch.no_grad():
        for _ in range(warmup):
            fn(*args)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            fn(*args)
        if device.type == "cuda":
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / reps

print("Setup ready.")

## Part 1: size sweep

Straight comparison at a realistic shape (8 heads, head_dim=64), sizes 16 up to 8192 -- mirrors how big a real attention call gets.

In [ ]:
torch.manual_seed(0)
D = 64
H = 8

print(f"{'size':>6} | {'naive':>12} {'fused(SDPA)':>13} | {'speedup':>8}")
for size, reps in [
    (16, 300), (32, 300), (64, 200), (128, 200), (256, 100),
    (512, 100), (1024, 50), (2048, 30), (4096, 15), (8192, 8),
]:
    q = torch.randn(1, H, size, D, device=device)
    k = torch.randn(1, H, size, D, device=device)
    v = torch.randn(1, H, size, D, device=device)
    sm_scale = 1 / sqrt(D)

    t_naive = bench(naive_attention, q, k, v, sm_scale, True, device=device, reps=reps)
    t_fused = bench(fused_attention, q, k, v, sm_scale, True, device=device, reps=reps)
    speedup = t_naive / t_fused
    print(f"{size:>6} | {t_naive*1e3:>11.4f}ms {t_fused*1e3:>12.4f}ms | {speedup:>7.2f}x")

## Part 2: isolation test -- this is the one that actually answers the question

Fixes the attention block size at B=16 (matches what our project's algorithm actually uses) and scales up M, the number of blocks processed in one call. Naive attention always launches exactly 3 GPU kernels (matmul, softmax, matmul) no matter how big M gets; the fused kernel always launches 1.

- If the speedup **shrinks toward/below 1x** as M grows: it's just fixed per-launch overhead being amortized away, not a real memory-bandwidth win (this is what we found on CPU).
- If the speedup **holds steady or grows** as M grows: that's the genuine FlashAttention effect -- the thing we couldn't test without a GPU.

In [ ]:
B = 16
sm_scale = 1 / sqrt(D)
print(f"{'M':>6} | {'naive':>12} {'fused':>12} | {'speedup':>8} | {'naive/block':>14} {'fused/block':>14}")
for M, reps in [
    (1, 400), (4, 400), (16, 300), (64, 200), (256, 100),
    (1024, 50), (4096, 20), (16384, 8), (65536, 4),
]:
    try:
        q = torch.randn(1, M, B, D, device=device)
        k = torch.randn(1, M, B, D, device=device)
        v = torch.randn(1, M, B, D, device=device)
    except RuntimeError as e:
        print(f"{M:>6} | (skipped -- out of GPU memory: {e})")
        continue
    t_naive = bench(naive_attention, q, k, v, sm_scale, True, device=device, reps=reps)
    t_fused = bench(fused_attention, q, k, v, sm_scale, True, device=device, reps=reps)
    speedup = t_naive / t_fused
    print(f"{M:>6} | {t_naive*1e3:>11.4f}ms {t_fused*1e3:>11.4f}ms | {speedup:>7.2f}x | "
          f"{t_naive/M*1e6:>11.4f}us {t_fused/M*1e6:>11.4f}us")

print()
print("Done -- please copy all output from both Part 1 and Part 2 cells above.")